# Spark DataFrames & Spark SQL 

## 1. Job, Stage və Task arasındakı fərqi izah et — hər biri nə vaxt yaranır?

Job o vaxt yaranir ki eger biz hansisa bir action vermisikse meselen show() yada count() cunki bunlar transformasiya emri deyil buna gorede lazy sayilmir belelikel job yaranir 

Stage eger biz groupby() yada join emrleri versek onda shuffle olur her shuffle olanda da stage yaranir data ele bil ki executer ler arasinda paylananda executerle butun datani gore bilsin deye bas verir
 
Task daki executerler ozu datanin partitionlarina esasen ozleri tasklara bolur sorada o tasklari paralel olarag icra edir 

## 2. Narrow və Wide transformation arasındakı fərqi izah et və hər birindən 2 nümunə (metod adı) ver.

Narrow transformation odur ki biz ele bir emr veririk ki burda partitonlar hamsi bir birini izlemesi lazim olmur meslen filter ve ya map emri veririk burda artig bir partitiondaki data hemin partitiona kifayet edir 

Wide transformation da ise biz ele bir emr veririk ki bu zaman mutleq partitionlar arasinda data mubadileesi olmalidir meselen biz groupby(country) emri vermisik bu zaman ele ola biler ki partition1 de az usa olsun partition 2 de ise usa ru olsun bu zaman netice olarag bize hamsi lazim olduguna gore mubadile yaranir ve biz hem az usa hemde ru setirlerinin neticesini goruruk

## 3. UDF ilə Built-in funksiya arasında performans fərqi niyə var?

Udf bizim ozumuzun yaratdigimiz funksiyalardi built in ise hazir funksiyalar ki bele ki biz funksiyani ozumuz yaradan orada spark optimization ede bilmir ve kod zeif process ola biler amma built in funksiyalarin her birini spark optimization edir ve daha suretli netice elde etmis olurug

In [1]:
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

# Python versiya uyğunsuzluğunu qarşısını almaq üçün mühit dəyişənləri
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 1. SparkSession-ın yaradılması
spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("local[*]")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

## 1. Hər iki faylı aydın (explicit) sxem təyin edərək Spark DataFrame kimi oxu və yüklə.

In [8]:
# 1. customers.csv faylını MinIO-dan explicit sxem ilə oxumaq
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("balance", DoubleType(), True),
    StructField("vip_status", StringType(), True)
])

df_customers = (
    spark.read
    .schema(customers_schema)
    .option("header", "true")
    .csv("s3a://bronze/customers.csv")
)

# 2. card_trn.csv faylını MinIO-dan explicit sxem ilə oxumaq
card_trn_schema = StructType([
    StructField("trn_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("trn_date", StringType(), True),
    StructField("merchant", StringType(), True)
])

df_card_trn = (
    spark.read
    .schema(card_trn_schema)
    .option("header", "true")
    .csv("s3a://bronze/card_trn.csv")
)

print("Datalar MinIO-dan uğurla oxundu!")
df_customers.show(2, truncate=False)
df_card_trn.show(2, truncate=False)

Datalar MinIO-dan uğurla oxundu!
+-----------+---------------+---+-------+----------+
|customer_id|customer_name  |age|balance|vip_status|
+-----------+---------------+---+-------+----------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |
|C002       |Sabina Mammadov|60 |3651.47|Y         |
+-----------+---------------+---+-------+----------+
only showing top 2 rows

+------+-----------+------+----------+--------+
|trn_id|customer_id|amount|trn_date  |merchant|
+------+-----------+------+----------+--------+
|T0001 |C001       |67.14 |2026-03-09|SOCAR   |
|T0002 |C001       |265.14|2026-07-11|Ecoteks |
+------+-----------+------+----------+--------+
only showing top 2 rows



## 2. Sütunları çevir: balance sütunu null olan sətirlərdə 0 ilə əvəz et.

In [ ]:
df_customers = df_customers.na.fill({"balance": 0.0})

print("Balance sütunu təmizləndi:")
df_customers.show(3, truncate=False)

Balance sütunu təmizləndi:
+-----------+---------------+---+-------+----------+
|customer_id|customer_name  |age|balance|vip_status|
+-----------+---------------+---+-------+----------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |
|C002       |Sabina Mammadov|60 |3651.47|Y         |
|C003       |Sevinj Karimova|19 |0.0    |Y         |
+-----------+---------------+---+-------+----------+
only showing top 3 rows



## 3. Yeni sütun əlavə et (age_group): age < 25 olarsa "young", 25-40 arası "adult", 40-dan yuxarı "senior".


In [ ]:
df_customers = df_customers.withColumn(
    "age_group",
    F.when(F.col("age") < 25, "young")
     .when((F.col("age") >= 25) & (F.col("age") <= 40), "adult")
     .otherwise("senior")
)

print("age_group sütunu əlavə olundu:")
df_customers.show(3, truncate=False)

age_group sütunu əlavə olundu:
+-----------+---------------+---+-------+----------+---------+
|customer_id|customer_name  |age|balance|vip_status|age_group|
+-----------+---------------+---+-------+----------+---------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |
+-----------+---------------+---+-------+----------+---------+
only showing top 3 rows



## 4. vip_status sütununun adını flg_is_vip olaraq dəyiş.


In [11]:
df_customers = df_customers.withColumnRenamed("vip_status", "flg_is_vip")

print("Sütun adı dəyişdirildi:")
df_customers.show(3, truncate=False)

Sütun adı dəyişdirildi:
+-----------+---------------+---+-------+----------+---------+
|customer_id|customer_name  |age|balance|flg_is_vip|age_group|
+-----------+---------------+---+-------+----------+---------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |
+-----------+---------------+---+-------+----------+---------+
only showing top 3 rows



## 5. Yeni sütun əlavə et: insert_date — bu, ETL-in icra tarixi olmalıdır. (Sysdate)


In [12]:
current_date_str = datetime.now().strftime("%Y-%m-%d")
df_customers = df_customers.withColumn("insert_date", F.lit(current_date_str))

print("insert_date sütunu əlavə edildi:")
df_customers.show(3, truncate=False)

insert_date sütunu əlavə edildi:
+-----------+---------------+---+-------+----------+---------+-----------+
|customer_id|customer_name  |age|balance|flg_is_vip|age_group|insert_date|
+-----------+---------------+---+-------+----------+---------+-----------+
|C001       |Rashad Aliyev  |23 |1230.15|Y         |young    |2026-09-11 |
|C002       |Sabina Mammadov|60 |3651.47|Y         |senior   |2026-09-11 |
|C003       |Sevinj Karimova|19 |0.0    |Y         |young    |2026-09-11 |
+-----------+---------------+---+-------+----------+---------+-----------+
only showing top 3 rows



## 6. Nəticəni silver bucket-ə "overwrite schema" seçimi ilə yaz.


In [ ]:
df_customers.write.format("parquet").mode("overwrite").option("overwriteSchema", "true").save("s3a://silver/customers")

print("Məlumatlar 'silver' bucket-ə uğurla yazıldı!")

Məlumatlar 'silver' bucket-ə uğurla yazıldı!


## 7. card_trn.csv üzərindən hər customer_id üçün total_amount hesabla.


In [19]:
df_customer_totals = df_card_trn.groupBy("customer_id").agg(
    F.sum("amount").alias("total_amount")
)

print("Hər müştəri ID-si üzrə toplam məbləğlər:")
df_customer_totals.show(5, truncate=False)

Hər müştəri ID-si üzrə toplam məbləğlər:
+-----------+------------------+
|customer_id|total_amount      |
+-----------+------------------+
|C006       |166.53            |
|C010       |251.5             |
|C007       |442.89            |
|C012       |641.71            |
|C003       |1471.8700000000001|
+-----------+------------------+
only showing top 5 rows



## 8. Nəticədə yalnız customer_name və total_amount sütunlarını göstər.


In [20]:
df_merged.select("customer_name", "total_amount").show(5, truncate=False)

+---------------+------------------+
|customer_name  |total_amount      |
+---------------+------------------+
|Farid Rzayev   |166.53            |
|Zeynab Abbasova|251.5             |
|Tural Karimova |442.89            |
|Sevinj Guliyev |641.71            |
|Sevinj Karimova|1471.8700000000001|
+---------------+------------------+
only showing top 5 rows



## 9. Nəticəni total_amount-a görə azalan sırada sırala və ilk 2 müştərini göstər.


In [21]:
df_merged.select("customer_name", "total_amount") \
         .orderBy(F.col("total_amount").desc()) \
         .show(2, truncate=False)

+---------------+------------------+
|customer_name  |total_amount      |
+---------------+------------------+
|Sevinj Karimova|1471.8700000000001|
|Gunel Mammadov |1471.15           |
+---------------+------------------+
only showing top 2 rows

